In [293]:
from pathlib import Path
import pandas as pd

# Paths (relative to notebooks/)
SYN_DIR = Path("../data/synthetic")
MEALS_PATH = SYN_DIR / "user_meals.csv"
SYMPTOMS_PATH = SYN_DIR / "user_symptoms.csv"

# Load
meals = pd.read_csv(MEALS_PATH)
symptoms = pd.read_csv(SYMPTOMS_PATH)

# Parse timestamps
meals["meal_time"] = pd.to_datetime(meals["meal_time"])
symptoms["symptom_time"] = pd.to_datetime(symptoms["symptom_time"])

# Quick sanity
print("Meals rows:", len(meals), "| unique meals:", meals["meal_id"].nunique())
print("Symptoms rows:", len(symptoms))
display(meals.head(10))
display(symptoms.head(10))


Meals rows: 664 | unique meals: 187
Symptoms rows: 32


,user_id,meal_id,meal_time,ingredient
0,u1,m1,2025-11-15 08:10:00,carrot
1,u1,m1,2025-11-15 08:10:00,honey
2,u1,m1,2025-11-15 08:10:00,oats
3,u1,m1,2025-11-15 08:10:00,potato
4,u1,m2,2025-11-15 13:13:00,banana
5,u1,m2,2025-11-15 13:13:00,chicken
6,u1,m2,2025-11-15 13:13:00,rice
7,u1,m2,2025-11-15 13:13:00,tomato
8,u1,m3,2025-11-15 19:29:00,banana
9,u1,m3,2025-11-15 19:29:00,pasta


,user_id,symptom_time,symptom_severity
0,u1,2025-11-15 12:25:00,2
1,u1,2025-11-21 11:48:00,2
2,u1,2025-11-23 23:27:00,4
3,u1,2025-11-28 20:40:00,2
4,u1,2025-12-01 17:00:00,4
5,u1,2025-12-03 23:59:00,4
6,u1,2025-12-04 07:23:00,1
7,u1,2025-12-04 14:11:00,5
8,u1,2025-12-09 11:51:00,5
9,u1,2025-12-09 18:38:00,2


In [294]:
# -----------------------------
# Trigger detection MVP config
# -----------------------------

# Case (lag) window: we treat meals in the last 0–24h before a symptom as "exposures"
CASE_HOURS = 24

# Control window: same clock-time on a different day to reduce daily-pattern bias
# (MVP: use 48–72h before the symptom)
CTRL_START_HOURS = 72
CTRL_END_HOURS   = 48

# Which symptoms count as "events" (MVP: use severity threshold)
EVENT_SEVERITY_THRESHOLD = 6

# Minimum evidence before we report an ingredient
MIN_CASE_EXPOSURES = 3

print("Config loaded:")
print("  CASE_HOURS =", CASE_HOURS)
print("  CTRL window =", f"[{CTRL_START_HOURS}, {CTRL_END_HOURS}] hours before symptom")
print("  EVENT_SEVERITY_THRESHOLD =", EVENT_SEVERITY_THRESHOLD)
print("  MIN_CASE_EXPOSURES =", MIN_CASE_EXPOSURES)


Config loaded:
  CASE_HOURS = 24
  CTRL window = [72, 48] hours before symptom
  EVENT_SEVERITY_THRESHOLD = 6
  MIN_CASE_EXPOSURES = 3


In [295]:
# -----------------------------
# Select symptom events
# -----------------------------

# We only treat high-severity symptoms as trigger "events"
events = symptoms[symptoms["symptom_severity"] >= EVENT_SEVERITY_THRESHOLD].copy()

events = events.sort_values("symptom_time").reset_index(drop=True)

print("Total symptoms:", len(symptoms))
print("Event symptoms (used for trigger detection):", len(events))

display(events)


Total symptoms: 32
Event symptoms (used for trigger detection): 7


,user_id,symptom_time,symptom_severity
0,u1,2026-01-02 22:30:00,8
1,u1,2026-01-06 22:40:00,8
2,u1,2026-01-06 23:40:00,7
3,u1,2026-01-11 23:10:00,8
4,u1,2026-01-16 22:55:00,7
5,u1,2026-01-17 03:55:00,7
6,u1,2026-01-17 04:55:00,7


In [296]:
# ------------------------------------------------
# Build case and control windows for each event
# ------------------------------------------------

windows = []

for _, row in events.iterrows():
    t_event = row["symptom_time"]

    windows.append({
        "event_time": t_event,

        # Case window: meals BEFORE symptom
        "case_start": t_event - pd.Timedelta(hours=CASE_HOURS),
        "case_end": t_event,

        # Control window: same clock-time, previous day
        "ctrl_start": t_event - pd.Timedelta(hours=CTRL_START_HOURS),
        "ctrl_end": t_event - pd.Timedelta(hours=CTRL_END_HOURS),
    })

windows_df = pd.DataFrame(windows)

print("Windows created:", len(windows_df))
display(windows_df.head())


Windows created: 7


,event_time,case_start,case_end,ctrl_start,ctrl_end
0,2026-01-02 22:30:00,2026-01-01 22:30:00,2026-01-02 22:30:00,2025-12-30 22:30:00,2025-12-31 22:30:00
1,2026-01-06 22:40:00,2026-01-05 22:40:00,2026-01-06 22:40:00,2026-01-03 22:40:00,2026-01-04 22:40:00
2,2026-01-06 23:40:00,2026-01-05 23:40:00,2026-01-06 23:40:00,2026-01-03 23:40:00,2026-01-04 23:40:00
3,2026-01-11 23:10:00,2026-01-10 23:10:00,2026-01-11 23:10:00,2026-01-08 23:10:00,2026-01-09 23:10:00
4,2026-01-16 22:55:00,2026-01-15 22:55:00,2026-01-16 22:55:00,2026-01-13 22:55:00,2026-01-14 22:55:00


In [297]:
# ------------------------------------------------
# Collapse meals into: one row per meal, list of ingredients
# ------------------------------------------------

# Expect meals to have: meal_id, meal_time, ingredient
required_cols = {"meal_id", "meal_time", "ingredient"}
missing_cols = required_cols - set(meals.columns)
if missing_cols:
    raise ValueError(f"user_meals.csv is missing columns: {missing_cols}")

# One row per meal_id, keep the meal_time, collect ingredients into a list
meals_by_meal = (
    meals.sort_values("meal_time")
         .groupby("meal_id", as_index=False)
         .agg(
             meal_time=("meal_time", "first"),
             ingredients=("ingredient", list),
         )
)

print("Raw meal rows:", len(meals))
print("Unique meals:", meals_by_meal["meal_id"].nunique())
display(meals_by_meal.head(10))


Raw meal rows: 664
Unique meals: 187


,meal_id,meal_time,ingredients
0,m1,2025-11-15 08:10:00,"[carrot, honey, oats, potato]"
1,m10,2025-11-18 08:04:00,"[oats, rice]"
2,m100,2025-12-21 08:12:00,"[oats, egg, potato, banana, cucumber]"
3,m101,2025-12-21 13:11:00,"[cucumber, rice, tomato]"
4,m102,2025-12-21 19:23:00,"[butter, flour, oats, pure olive oil]"
5,m103,2025-12-22 08:04:00,"[pepper, oats, milk, honey]"
6,m104,2025-12-22 13:01:00,"[beans, chicken, cucumber]"
7,m105,2025-12-22 19:38:00,"[banana, tomato]"
8,m106,2025-12-23 08:08:00,"[honey, egg, cucumber, beans, banana]"
9,m107,2025-12-23 13:09:00,"[apple, beans, onion, pure olive oil]"


In [298]:
# ------------------------------------------------
# For each event: collect case/control ingredients
# ------------------------------------------------

def ingredients_in_window(meals_by_meal: pd.DataFrame, start, end):
    """Return a flat list of ingredients for meals with meal_time in [start, end)."""
    m = meals_by_meal[(meals_by_meal["meal_time"] >= start) & (meals_by_meal["meal_time"] < end)]
    flat = []
    for lst in m["ingredients"]:
        flat.extend(lst)
    return flat

event_rows = []

for idx, w in windows_df.iterrows():
    case_ings = ingredients_in_window(meals_by_meal, w["case_start"], w["case_end"])
    ctrl_ings = ingredients_in_window(meals_by_meal, w["ctrl_start"], w["ctrl_end"])

    event_rows.append({
        "event_id": idx,
        "event_time": w["event_time"],
        "case_start": w["case_start"],
        "case_end": w["case_end"],
        "ctrl_start": w["ctrl_start"],
        "ctrl_end": w["ctrl_end"],
        "case_ingredients": case_ings,
        "ctrl_ingredients": ctrl_ings,
        "n_case_meals": int(((meals_by_meal["meal_time"] >= w["case_start"]) & (meals_by_meal["meal_time"] < w["case_end"])).sum()),
        "n_ctrl_meals": int(((meals_by_meal["meal_time"] >= w["ctrl_start"]) & (meals_by_meal["meal_time"] < w["ctrl_end"])).sum()),
    })

event_exposures = pd.DataFrame(event_rows)

print("Built event_exposures:", len(event_exposures), "events")
display(event_exposures[["event_id", "event_time", "n_case_meals", "n_ctrl_meals", "case_ingredients", "ctrl_ingredients"]])


Built event_exposures: 7 events


,event_id,event_time,n_case_meals,n_ctrl_meals,case_ingredients,ctrl_ingredients
0,0,2026-01-02 22:30:00,3,3,"[beans, egg, honey, milk, pure olive oil, bean...","[oats, rice, lettuce, apple, egg, beans, lettu..."
1,1,2026-01-06 22:40:00,3,2,"[oats, honey, egg, cucumber, beans, banana, pe...","[apple, banana, cucumber, tomato, apple, chick..."
2,2,2026-01-06 23:40:00,3,2,"[oats, honey, egg, cucumber, beans, banana, pe...","[apple, banana, cucumber, tomato, apple, chick..."
3,3,2026-01-11 23:10:00,3,3,"[apple, banana, chicken, oats, cucumber, tomat...","[cucumber, oats, potato, banana, beans, pure o..."
4,4,2026-01-16 22:55:00,3,2,"[banana, honey, lettuce, milk, potato, onion, ...","[carrot, honey, pure olive oil, tomato, pepper..."
5,5,2026-01-17 03:55:00,3,2,"[banana, honey, lettuce, milk, potato, onion, ...","[carrot, honey, pure olive oil, tomato, pepper..."
6,6,2026-01-17 04:55:00,3,2,"[banana, honey, lettuce, milk, potato, onion, ...","[carrot, honey, pure olive oil, tomato, pepper..."


In [299]:
# ------------------------------------------------
# Count ingredient appearances in case vs control windows
# ------------------------------------------------

from collections import Counter

case_counts = Counter()
ctrl_counts = Counter()

for _, row in event_exposures.iterrows():
    # Use "set" so an ingredient counts at most once per event window
    # (prevents one day with 3 cheese meals dominating)
    case_counts.update(set(row["case_ingredients"]))
    ctrl_counts.update(set(row["ctrl_ingredients"]))

all_ings = sorted(set(case_counts.keys()) | set(ctrl_counts.keys()))

counts_df = pd.DataFrame({
    "ingredient": all_ings,
    "case_events": [case_counts.get(i, 0) for i in all_ings],
    "ctrl_events": [ctrl_counts.get(i, 0) for i in all_ings],
})

# Quick filters: remove ingredients with basically no evidence
counts_df["total_events_seen"] = counts_df["case_events"] + counts_df["ctrl_events"]

print("Unique ingredients seen in any window:", len(counts_df))
display(counts_df.sort_values("case_events", ascending=False).head(20))


Unique ingredients seen in any window: 20


,ingredient,case_events,ctrl_events,total_events_seen
9,honey,7,3,10
10,lettuce,6,2,8
1,banana,6,4,10
16,potato,5,1,6
4,cheese,5,0,5
2,beans,4,2,6
3,carrot,4,3,7
11,milk,4,0,4
18,rice,3,5,8
7,cucumber,3,3,6


In [300]:
# ------------------------------------------------
# Compute smoothed case–control trigger score
# ------------------------------------------------

import numpy as np

# Small smoothing constant (Laplace smoothing)
ALPHA = 0.5

# Total number of events
N_EVENTS = len(event_exposures)

scores = []

for _, row in counts_df.iterrows():
    c_case = row["case_events"]
    c_ctrl = row["ctrl_events"]

    # Smoothed probabilities
    p_case = (c_case + ALPHA) / (N_EVENTS + 2 * ALPHA)
    p_ctrl = (c_ctrl + ALPHA) / (N_EVENTS + 2 * ALPHA)

    # Log odds ratio (core statistical signal)
    log_odds = np.log(p_case / p_ctrl)

    scores.append({
        "ingredient": row["ingredient"],
        "case_events": c_case,
        "ctrl_events": c_ctrl,
        "log_odds": log_odds,
    })

scores_df = pd.DataFrame(scores)

scores_df = scores_df.sort_values("log_odds", ascending=False).reset_index(drop=True)

display(scores_df)


,ingredient,case_events,ctrl_events,log_odds
0,cheese,5,0,2.397895
1,milk,4,0,2.197225
2,potato,5,1,1.299283
3,cream,1,0,1.098612
4,lettuce,6,2,0.955511
5,onion,3,1,0.847298
6,egg,3,1,0.847298
7,honey,7,3,0.762140
8,beans,4,2,0.587787
9,banana,6,4,0.367725


In [301]:
# ------------------------------------------------
# Evidence weighting (Empirical Bayes–style shrinkage)
# ------------------------------------------------

# Intuition:
# - Ingredients seen only once or twice are unreliable → shrink heavily
# - Ingredients seen many times are reliable → shrink very little

# Strength of the prior (how cautious we are)
# Larger = more conservative (MVP default is good)
PRIOR_STRENGTH = 5.0

shrunken_rows = []

for _, row in scores_df.iterrows():
    total_evidence = row["case_events"] + row["ctrl_events"]

    # Weight ∈ [0, 1]: how much we trust this ingredient
    # Goes up smoothly with more evidence
    weight = total_evidence / (total_evidence + PRIOR_STRENGTH)

    # Shrink log-odds toward 0 (no effect)
    shrunken_log_odds = weight * row["log_odds"]

    shrunken_rows.append({
        "ingredient": row["ingredient"],
        "case_events": row["case_events"],
        "ctrl_events": row["ctrl_events"],
        "total_events": total_evidence,
        "raw_log_odds": row["log_odds"],
        "evidence_weight": weight,
        "shrunken_log_odds": shrunken_log_odds,
    })

shrunken_df = (
    pd.DataFrame(shrunken_rows)
      .sort_values("shrunken_log_odds", ascending=False)
      .reset_index(drop=True)
)

display(shrunken_df)


,ingredient,case_events,ctrl_events,total_events,raw_log_odds,evidence_weight,shrunken_log_odds
0,cheese,5,0,5,2.397895,0.500000,1.198948
1,milk,4,0,4,2.197225,0.444444,0.976544
2,potato,5,1,6,1.299283,0.545455,0.708700
3,lettuce,6,2,8,0.955511,0.615385,0.588007
4,honey,7,3,10,0.762140,0.666667,0.508093
5,onion,3,1,4,0.847298,0.444444,0.376577
6,egg,3,1,4,0.847298,0.444444,0.376577
7,beans,4,2,6,0.587787,0.545455,0.320611
8,banana,6,4,10,0.367725,0.666667,0.245150
9,cream,1,0,1,1.098612,0.166667,0.183102


In [302]:
# ------------------------------------------------
# Aggregate trigger signal at ingredient-family level
# ------------------------------------------------

# Load ingredient -> family mapping
GROUPS_PATH = Path("../data/processed") / "ingredient_groups.csv"
groups_df = pd.read_csv(GROUPS_PATH)

# Build ingredient -> group_head dictionary
ingredient_to_group = dict(
    zip(groups_df["ingredient"].astype(str), groups_df["group_head"].astype(str))
)

# Map ingredient to family (fallback: ingredient itself)
shrunken_df["family"] = shrunken_df["ingredient"].map(
    lambda x: ingredient_to_group.get(x, x)
)

# Aggregate at family level
family_df = (
    shrunken_df
        .groupby("family", as_index=False)
        .agg(
            family_score=("shrunken_log_odds", "sum"),
            family_evidence=("total_events", "sum"),
            n_members=("ingredient", "count"),
        )
        .sort_values("family_score", ascending=False)
        .reset_index(drop=True)
)

display(family_df.head(15))


,family,family_score,family_evidence,n_members
0,cheese,1.198948,5,1
1,milk,0.976544,4,1
2,potato,0.708700,6,1
3,lettuce,0.588007,8,1
4,honey,0.508093,10,1
5,egg,0.376577,4,1
6,onion,0.376577,4,1
7,beans,0.320611,6,1
8,banana,0.245150,10,1
9,cream,0.183102,1,1


In [303]:
# ------------------------------------------------
# Final MVP trigger ranking (clean, interpretable)
# ------------------------------------------------

# Filter: require minimum evidence
final_df = family_df[family_df["family_evidence"] >= MIN_CASE_EXPOSURES].copy()

# If nothing survives (synthetic data can be small), fall back gracefully
if final_df.empty:
    print("Warning: no families meet MIN_CASE_EXPOSURES — showing all results")
    final_df = family_df.copy()

# Normalise scores to [0, 1] for interpretability
max_score = final_df["family_score"].abs().max()
if max_score > 0:
    final_df["trigger_strength"] = final_df["family_score"] / max_score
else:
    final_df["trigger_strength"] = 0.0

# Rank (1 = strongest trigger)
final_df["rank"] = (
    final_df["trigger_strength"]
        .rank(ascending=False, method="dense")
        .astype(int)
)

# Sort cleanly
final_df = (
    final_df
        .sort_values("rank")
        .reset_index(drop=True)
)

# Select only what the MVP needs
final_triggers = final_df[[
    "rank",
    "family",
    "trigger_strength",
    "family_evidence",
    "n_members",
]]

display(final_triggers)


,rank,family,trigger_strength,family_evidence,n_members
0,1,cheese,1.000000,5,1
1,2,milk,0.814501,4,1
2,3,potato,0.591102,6,1
3,4,lettuce,0.490436,8,1
4,5,honey,0.423783,10,1
5,6,egg,0.314089,4,1
6,6,onion,0.314089,4,1
7,7,beans,0.267410,6,1
8,8,banana,0.204471,10,1
9,9,oats,0.140320,5,1


In [304]:
# ------------------------------------------------
# Improved v1: cluster symptom logs into "episodes"
# (prevents one flare being counted multiple times)
# ------------------------------------------------

# If two high-severity symptom logs are within this many hours,
# treat them as the SAME episode
EPISODE_GAP_HOURS = 8

# Sort symptoms
sym_sorted = symptoms.sort_values("symptom_time").reset_index(drop=True)

episodes = []
current = None
gap = pd.Timedelta(hours=EPISODE_GAP_HOURS)

for _, r in sym_sorted.iterrows():
    if r["symptom_severity"] < EVENT_SEVERITY_THRESHOLD:
        continue

    t = r["symptom_time"]
    sev = float(r["symptom_severity"])

    if current is None:
        current = {"start_time": t, "end_time": t, "max_severity": sev}
        episodes.append(current)
        continue

    # Same episode if close in time
    if t - current["end_time"] <= gap:
        current["end_time"] = t
        if sev > current["max_severity"]:
            current["max_severity"] = sev
    else:
        current = {"start_time": t, "end_time": t, "max_severity": sev}
        episodes.append(current)

episodes_df = pd.DataFrame(episodes)

# Define the event time as the episode start
episodes_df["event_time"] = episodes_df["start_time"]

print("High-severity symptom logs:", int((symptoms["symptom_severity"] >= EVENT_SEVERITY_THRESHOLD).sum()))
print("Clustered episodes:", len(episodes_df))

display(episodes_df[["event_time", "start_time", "end_time", "max_severity"]])


High-severity symptom logs: 7
Clustered episodes: 4


,event_time,start_time,end_time,max_severity
0,2026-01-02 22:30:00,2026-01-02 22:30:00,2026-01-02 22:30:00,8.0
1,2026-01-06 22:40:00,2026-01-06 22:40:00,2026-01-06 23:40:00,8.0
2,2026-01-11 23:10:00,2026-01-11 23:10:00,2026-01-11 23:10:00,8.0
3,2026-01-16 22:55:00,2026-01-16 22:55:00,2026-01-17 04:55:00,7.0


In [305]:
# ------------------------------------------------
# Use EPISODES instead of raw symptom logs
# (drop-in replacement for events)
# ------------------------------------------------

# Replace events with episode onsets
events = episodes_df[["event_time"]].copy()

events = events.sort_values("event_time").reset_index(drop=True)

print("Episode-based events used for trigger detection:", len(events))
display(events)

# ------------------------------------------------
# Rebuild case & control windows from episodes
# ------------------------------------------------

windows = []

for _, row in events.iterrows():
    t_event = row["event_time"]

    windows.append({
        "event_time": t_event,

        # Case window: meals BEFORE episode onset
        "case_start": t_event - pd.Timedelta(hours=CASE_HOURS),
        "case_end": t_event,

        # Control window: same clock-time, previous day
        "ctrl_start": t_event - pd.Timedelta(hours=CTRL_START_HOURS),
        "ctrl_end": t_event - pd.Timedelta(hours=CTRL_END_HOURS),
    })

windows_df = pd.DataFrame(windows)

print("Episode-based windows created:", len(windows_df))
display(windows_df)


Episode-based events used for trigger detection: 4


,event_time
0,2026-01-02 22:30:00
1,2026-01-06 22:40:00
2,2026-01-11 23:10:00
3,2026-01-16 22:55:00


Episode-based windows created: 4


,event_time,case_start,case_end,ctrl_start,ctrl_end
0,2026-01-02 22:30:00,2026-01-01 22:30:00,2026-01-02 22:30:00,2025-12-30 22:30:00,2025-12-31 22:30:00
1,2026-01-06 22:40:00,2026-01-05 22:40:00,2026-01-06 22:40:00,2026-01-03 22:40:00,2026-01-04 22:40:00
2,2026-01-11 23:10:00,2026-01-10 23:10:00,2026-01-11 23:10:00,2026-01-08 23:10:00,2026-01-09 23:10:00
3,2026-01-16 22:55:00,2026-01-15 22:55:00,2026-01-16 22:55:00,2026-01-13 22:55:00,2026-01-14 22:55:00


In [306]:
# ------------------------------------------------
# Rebuild event_exposures from EPISODE windows_df
# ------------------------------------------------

event_rows = []

for idx, w in windows_df.iterrows():
    case_ings = ingredients_in_window(meals_by_meal, w["case_start"], w["case_end"])
    ctrl_ings = ingredients_in_window(meals_by_meal, w["ctrl_start"], w["ctrl_end"])

    n_case_meals = int(((meals_by_meal["meal_time"] >= w["case_start"]) & (meals_by_meal["meal_time"] < w["case_end"])).sum())
    n_ctrl_meals = int(((meals_by_meal["meal_time"] >= w["ctrl_start"]) & (meals_by_meal["meal_time"] < w["ctrl_end"])).sum())

    event_rows.append({
        "event_id": idx,
        "event_time": w["event_time"],
        "case_start": w["case_start"],
        "case_end": w["case_end"],
        "ctrl_start": w["ctrl_start"],
        "ctrl_end": w["ctrl_end"],
        "case_ingredients": case_ings,
        "ctrl_ingredients": ctrl_ings,
        "n_case_meals": n_case_meals,
        "n_ctrl_meals": n_ctrl_meals,
    })

event_exposures = pd.DataFrame(event_rows)

print("Built EPISODE-based event_exposures:", len(event_exposures), "events")
display(event_exposures[[
    "event_id", "event_time", "n_case_meals", "n_ctrl_meals", "case_ingredients", "ctrl_ingredients"
]])


Built EPISODE-based event_exposures: 4 events


,event_id,event_time,n_case_meals,n_ctrl_meals,case_ingredients,ctrl_ingredients
0,0,2026-01-02 22:30:00,3,3,"[beans, egg, honey, milk, pure olive oil, bean...","[oats, rice, lettuce, apple, egg, beans, lettu..."
1,1,2026-01-06 22:40:00,3,2,"[oats, honey, egg, cucumber, beans, banana, pe...","[apple, banana, cucumber, tomato, apple, chick..."
2,2,2026-01-11 23:10:00,3,3,"[apple, banana, chicken, oats, cucumber, tomat...","[cucumber, oats, potato, banana, beans, pure o..."
3,3,2026-01-16 22:55:00,3,2,"[banana, honey, lettuce, milk, potato, onion, ...","[carrot, honey, pure olive oil, tomato, pepper..."


In [307]:
# ------------------------------------------------
# Re-run trigger scoring on EPISODE-based event_exposures
# ------------------------------------------------

from collections import Counter
import numpy as np

# 1) Count ingredient appearances (once per event window)
case_counts = Counter()
ctrl_counts = Counter()

for _, row in event_exposures.iterrows():
    case_counts.update(set(row["case_ingredients"]))
    ctrl_counts.update(set(row["ctrl_ingredients"]))

all_ings = sorted(set(case_counts.keys()) | set(ctrl_counts.keys()))

counts_df = pd.DataFrame({
    "ingredient": all_ings,
    "case_events": [case_counts.get(i, 0) for i in all_ings],
    "ctrl_events": [ctrl_counts.get(i, 0) for i in all_ings],
})
counts_df["total_events_seen"] = counts_df["case_events"] + counts_df["ctrl_events"]

# 2) Smoothed log-odds
N_EVENTS = len(event_exposures)

scores = []
for _, r in counts_df.iterrows():
    c_case = int(r["case_events"])
    c_ctrl = int(r["ctrl_events"])

    p_case = (c_case + ALPHA) / (N_EVENTS + 2 * ALPHA)
    p_ctrl = (c_ctrl + ALPHA) / (N_EVENTS + 2 * ALPHA)

    scores.append({
        "ingredient": r["ingredient"],
        "case_events": c_case,
        "ctrl_events": c_ctrl,
        "total_events": c_case + c_ctrl,
        "raw_log_odds": float(np.log(p_case / p_ctrl)),
    })

scores_df = pd.DataFrame(scores).sort_values("raw_log_odds", ascending=False).reset_index(drop=True)

# 3) Evidence shrinkage
shrunken_rows = []
for _, r in scores_df.iterrows():
    total_evidence = int(r["total_events"])
    weight = total_evidence / (total_evidence + PRIOR_STRENGTH)
    shrunken_rows.append({
        "ingredient": r["ingredient"],
        "case_events": int(r["case_events"]),
        "ctrl_events": int(r["ctrl_events"]),
        "total_events": total_evidence,
        "raw_log_odds": float(r["raw_log_odds"]),
        "evidence_weight": float(weight),
        "shrunken_log_odds": float(weight * r["raw_log_odds"]),
    })

shrunken_df = pd.DataFrame(shrunken_rows).sort_values("shrunken_log_odds", ascending=False).reset_index(drop=True)

# 4) Map to family + aggregate
shrunken_df["family"] = shrunken_df["ingredient"].map(lambda x: ingredient_to_group.get(x, x))

family_df = (
    shrunken_df
        .groupby("family", as_index=False)
        .agg(
            family_score=("shrunken_log_odds", "sum"),
            family_evidence=("total_events", "sum"),
            n_members=("ingredient", "count"),
        )
        .sort_values("family_score", ascending=False)
        .reset_index(drop=True)
)

# 5) Final ranking (episode-based) — WITH true "min case exposures" filter

# Add family-level case/control counts (so evidence is interpretable)
family_counts = (
    shrunken_df
        .groupby("family", as_index=False)
        .agg(
            family_case_events=("case_events", "sum"),
            family_ctrl_events=("ctrl_events", "sum"),
            family_score=("shrunken_log_odds", "sum"),
            n_members=("ingredient", "count"),
        )
)

family_counts["family_total_events"] = (
    family_counts["family_case_events"] + family_counts["family_ctrl_events"]
)

family_counts = family_counts.sort_values("family_score", ascending=False).reset_index(drop=True)

# Filter using TRUE case evidence
final_df = family_counts[family_counts["family_case_events"] >= MIN_CASE_EXPOSURES].copy()
if final_df.empty:
    print("Warning: no families meet MIN_CASE_EXPOSURES (case events) — showing all results")
    final_df = family_counts.copy()

# Normalise to [0, 1]
max_score = final_df["family_score"].abs().max()
final_df["trigger_strength"] = final_df["family_score"] / max_score if max_score > 0 else 0.0

final_df["rank"] = final_df["trigger_strength"].rank(ascending=False, method="dense").astype(int)

final_triggers_episode = (
    final_df
        .sort_values("rank")
        .reset_index(drop=True)[[
            "rank",
            "family",
            "trigger_strength",
            "family_case_events",
            "family_ctrl_events",
            "family_total_events",
            "n_members",
        ]]
)

print("Episode-based trigger ranking:")
display(final_triggers_episode)


Episode-based trigger ranking:


,rank,family,trigger_strength,family_case_events,family_ctrl_events,family_total_events,n_members
0,1,honey,1.00000,4,1,5,1
1,2,lettuce,0.30627,3,2,5,1
2,2,beans,0.30627,3,2,5,1
3,3,banana,0.00000,3,3,6,1


In [308]:
# ------------------------------------------------
# Sanity check: compare MVP vs Episode-based rankings
# (so you can see what changed after episode clustering)
# ------------------------------------------------

# MVP (old table): rank, family, trigger_strength, family_evidence, n_members
mvp = final_triggers.copy()

# Episode-based (new table): rank, family, trigger_strength, family_case_events, family_ctrl_events, family_total_events, n_members
epi = final_triggers_episode.copy()

# ---- Rename columns for clarity (handle both schemas safely)

mvp = mvp.rename(columns={
    "rank": "mvp_rank",
    "trigger_strength": "mvp_strength",
    "n_members": "mvp_members",
})

# MVP might have family_evidence (old)
if "family_evidence" in mvp.columns:
    mvp = mvp.rename(columns={"family_evidence": "mvp_total_events"})

epi = epi.rename(columns={
    "rank": "epi_rank",
    "trigger_strength": "epi_strength",
    "n_members": "epi_members",
})

# Episode table has the new evidence columns
# (keep names, just ensure they're present)
for col in ["family_case_events", "family_ctrl_events", "family_total_events"]:
    if col not in epi.columns:
        epi[col] = 0

# ---- Merge (outer join so we keep families that only appear in one method)
compare_df = pd.merge(mvp, epi, on="family", how="outer")

# ---- Fill missing defaults
fill_defaults = {
    "mvp_rank": 999,
    "mvp_strength": 0.0,
    "mvp_members": 0,
    "mvp_total_events": 0,   # if MVP had family_evidence; if not, we'll add below
    "epi_rank": 999,
    "epi_strength": 0.0,
    "epi_members": 0,
    "family_case_events": 0,
    "family_ctrl_events": 0,
    "family_total_events": 0,
}
for k, v in fill_defaults.items():
    if k not in compare_df.columns:
        compare_df[k] = v
compare_df = compare_df.fillna(fill_defaults)

# If MVP didn't have total events, keep it as 0 (already handled above)
# ---- Deltas (negative rank_change means "episode method ranks it higher")
compare_df["rank_change"] = compare_df["epi_rank"].astype(int) - compare_df["mvp_rank"].astype(int)
compare_df["strength_change"] = compare_df["epi_strength"] - compare_df["mvp_strength"]

# Evidence change: compare episode TOTAL events vs MVP total events (if MVP had it)
compare_df["total_events_change"] = (
    compare_df["family_total_events"].astype(int) - compare_df["mvp_total_events"].astype(int)
)

# Also show case/control evidence deltas (episode has these; MVP doesn't)
compare_df["epi_case_events"] = compare_df["family_case_events"].astype(int)
compare_df["epi_ctrl_events"] = compare_df["family_ctrl_events"].astype(int)

# Sort: biggest rank improvements first
compare_df = compare_df.sort_values(
    by=["rank_change", "epi_rank", "mvp_rank"],
    ascending=[True, True, True]
).reset_index(drop=True)

print("Comparison table (Episode vs MVP):")
display(compare_df[[
    "family",
    "mvp_rank", "epi_rank", "rank_change",
    "mvp_strength", "epi_strength", "strength_change",
    "mvp_total_events", "family_total_events", "total_events_change",
    "epi_case_events", "epi_ctrl_events",
    "mvp_members", "epi_members",
]].head(25))

print("\nTop Episode-based triggers:")
display(epi.sort_values("epi_rank").head(15))

print("\nTop MVP triggers:")
display(mvp.sort_values("mvp_rank").head(15))


Comparison table (Episode vs MVP):


,family,mvp_rank,epi_rank,rank_change,mvp_strength,epi_strength,strength_change,mvp_total_events,family_total_events,total_events_change,epi_case_events,epi_ctrl_events,mvp_members,epi_members
0,beans,7,2.0,-5,0.267410,0.30627,0.038860,6,5.0,-1,3,2,1,1.0
1,banana,8,3.0,-5,0.204471,0.00000,-0.204471,10,6.0,-4,3,3,1,1.0
2,honey,5,1.0,-4,0.423783,1.00000,0.576217,10,5.0,-5,4,1,1,1.0
3,lettuce,4,2.0,-2,0.490436,0.30627,-0.184166,8,5.0,-3,3,2,1,1.0
4,tomato,16,999.0,983,-0.591102,0.00000,0.591102,6,0.0,-6,0,0,1,0.0
5,apple,15,999.0,984,-0.458157,0.00000,0.458157,5,0.0,-5,0,0,1,0.0
6,oil,14,999.0,985,-0.331919,0.00000,0.331919,9,0.0,-9,0,0,1,0.0
7,pepper,13,999.0,986,-0.267410,0.00000,0.267410,6,0.0,-6,0,0,1,0.0
8,rice,12,999.0,987,-0.231991,0.00000,0.231991,8,0.0,-8,0,0,1,0.0
9,chicken,11,999.0,988,0.000000,0.00000,0.000000,4,0.0,-4,0,0,1,0.0



Top Episode-based triggers:


,epi_rank,family,epi_strength,family_case_events,family_ctrl_events,family_total_events,epi_members
0,1,honey,1.00000,4,1,5,1
1,2,lettuce,0.30627,3,2,5,1
2,2,beans,0.30627,3,2,5,1
3,3,banana,0.00000,3,3,6,1



Top MVP triggers:


,mvp_rank,family,mvp_strength,mvp_total_events,mvp_members
0,1,cheese,1.000000,5,1
1,2,milk,0.814501,4,1
2,3,potato,0.591102,6,1
3,4,lettuce,0.490436,8,1
4,5,honey,0.423783,10,1
5,6,egg,0.314089,4,1
6,6,onion,0.314089,4,1
7,7,beans,0.267410,6,1
8,8,banana,0.204471,10,1
9,9,oats,0.140320,5,1


In [309]:
# ------------------------------------------------
# Export: final episode-based trigger tables (for backend + dissertation)
# ------------------------------------------------

OUT_DIR = Path("../data/processed") / "trigger_detection"
OUT_DIR.mkdir(parents=True, exist_ok=True)

run_meta = {
    "method": "episode_case_control_logodds_shrinkage",
    "case_hours": CASE_HOURS,
    "ctrl_start_hours": CTRL_START_HOURS,
    "ctrl_end_hours": CTRL_END_HOURS,
    "event_severity_threshold": EVENT_SEVERITY_THRESHOLD,
    "episode_gap_hours": EPISODE_GAP_HOURS,
    "alpha": ALPHA,
    "prior_strength": PRIOR_STRENGTH,
    "min_case_exposures": MIN_CASE_EXPOSURES,
}

# Ingredient-level (episode-based)
ingredient_out = shrunken_df.copy()
for k, v in run_meta.items():
    ingredient_out[k] = v

# Family-level + final ranking (episode-based)
family_out = final_triggers_episode.copy()
for k, v in run_meta.items():
    family_out[k] = v

ingredient_path = OUT_DIR / "episode_ingredient_scores.csv"
family_path = OUT_DIR / "episode_family_triggers.csv"

ingredient_out.to_csv(ingredient_path, index=False)
family_out.to_csv(family_path, index=False)

display(pd.DataFrame([{
    "saved": "ingredient-level",
    "rows": len(ingredient_out),
    "path": str(ingredient_path),
}]))

display(pd.DataFrame([{
    "saved": "family-level",
    "rows": len(family_out),
    "path": str(family_path),
}]))


,saved,rows,path
0,ingredient-level,20,../data/processed/trigger_detection/episode_in...


,saved,rows,path
0,family-level,4,../data/processed/trigger_detection/episode_fa...


In [310]:
# ============================================================
# Recommended Primary Method (MVP-safe):
# Time-stratified case-crossover with MULTIPLE matched controls
# + Discordant-pairs OR (U/V) + 95% CI + BH-FDR
# ============================================================

import numpy as np
import pandas as pd
from math import erf, sqrt
from collections import defaultdict

# -----------------------------
# Controls configuration
# -----------------------------
# Recommended: 3–4 matched controls per event (same clock-time, other days) :contentReference[oaicite:10]{index=10}
CONTROL_DAY_LAGS = [2, 3, 4, 7]   # you can change, but keep 3–4 controls minimum

# Symptom-free requirement for a control window:
# exclude a control if ANY high-severity episode starts within its case window
# (so controls are genuinely "non-event" periods)
def is_symptom_free_window(window_start, window_end, episode_event_times: pd.Series) -> bool:
    # True if no episode onset in [window_start, window_end)
    m = (episode_event_times >= window_start) & (episode_event_times < window_end)
    return not bool(m.any())

# We will use episode onsets if you have episodes_df; otherwise fall back to high severity symptom logs
if "episodes_df" in globals() and len(episodes_df) > 0:
    episode_times = pd.to_datetime(episodes_df["event_time"])
else:
    episode_times = pd.to_datetime(
        symptoms.loc[symptoms["symptom_severity"] >= EVENT_SEVERITY_THRESHOLD, "symptom_time"]
    )

episode_times = episode_times.sort_values().reset_index(drop=True)

# -----------------------------
# Build strata: each event has 1 case window + K matched control windows
# -----------------------------
# events should already be episode-based events with column "event_time"
if "events" not in globals() or "event_time" not in events.columns:
    raise ValueError("Expected `events` with column `event_time` (episode-based). Run the previous episode cells first.")

events = events.copy()
events["event_time"] = pd.to_datetime(events["event_time"])
events = events.sort_values("event_time").reset_index(drop=True)

strata_rows = []

for eid, r in events.iterrows():
    t_event = r["event_time"]

    # Case window for this event
    case_start = t_event - pd.Timedelta(hours=CASE_HOURS)
    case_end   = t_event

    # Controls: same clock-time, earlier days, symptom-free
    controls = []
    for d in CONTROL_DAY_LAGS:
        ctrl_end = t_event - pd.Timedelta(days=d)
        ctrl_start = ctrl_end - pd.Timedelta(hours=CASE_HOURS)

        # Must be symptom-free to count as a control stratum :contentReference[oaicite:11]{index=11}
        if is_symptom_free_window(ctrl_start, ctrl_end, episode_times):
            controls.append((ctrl_start, ctrl_end))

    # Keep at least 1 control; ideally 3–4
    if len(controls) == 0:
        # fallback: keep your old 48–72h control window if nothing qualifies (synthetic can be sparse)
        fallback_ctrl_start = t_event - pd.Timedelta(hours=CTRL_START_HOURS)
        fallback_ctrl_end   = t_event - pd.Timedelta(hours=CTRL_END_HOURS)
        controls.append((fallback_ctrl_start, fallback_ctrl_end))

    strata_rows.append({
        "event_id": eid,
        "event_time": t_event,
        "case_start": case_start,
        "case_end": case_end,
        "control_windows": controls,   # list of (start,end)
    })

strata_df = pd.DataFrame(strata_rows)

print("Strata built:", len(strata_df))
print("Controls per event (min/median/max):",
      int(strata_df["control_windows"].map(len).min()),
      float(strata_df["control_windows"].map(len).median()),
      int(strata_df["control_windows"].map(len).max()))
display(strata_df.head())

# -----------------------------
# Helper: collect ingredients in a time window from meals_by_meal
# -----------------------------
def ingredients_set_in_window(meals_by_meal: pd.DataFrame, start, end) -> set:
    m = meals_by_meal[(meals_by_meal["meal_time"] >= start) & (meals_by_meal["meal_time"] < end)]
    out = set()
    for lst in m["ingredients"]:
        out.update(lst)
    return out

# Ensure meals_by_meal exists
if "meals_by_meal" not in globals():
    raise ValueError("Expected `meals_by_meal`. Run the earlier 'Collapse meals' cell first.")

# -----------------------------
# Build per-stratum sets (case set, union control set)
# -----------------------------
case_sets = []
control_union_sets = []

for _, s in strata_df.iterrows():
    case_set = ingredients_set_in_window(meals_by_meal, s["case_start"], s["case_end"])

    ctrl_union = set()
    for (cs, ce) in s["control_windows"]:
        ctrl_union |= ingredients_set_in_window(meals_by_meal, cs, ce)

    case_sets.append(case_set)
    control_union_sets.append(ctrl_union)

strata_df["case_set"] = case_sets
strata_df["control_union_set"] = control_union_sets

display(strata_df[["event_id", "event_time", "case_set", "control_union_set"]].head())

# -----------------------------
# Discordant-pairs counting (U/V) per ingredient
# U_i: present in case but absent in ALL controls
# V_i: absent in case but present in >=1 control
# OR ≈ U/V (with pseudocount) :contentReference[oaicite:12]{index=12}
# -----------------------------
U = defaultdict(int)
V = defaultdict(int)

for _, s in strata_df.iterrows():
    case_set = s["case_set"]
    ctrl_set = s["control_union_set"]

    # discordant contributions
    for ing in (case_set - ctrl_set):
        U[ing] += 1
    for ing in (ctrl_set - case_set):
        V[ing] += 1

all_ings = sorted(set(U.keys()) | set(V.keys()))

# Normal CDF without scipy (for p-values)
def norm_cdf(z: float) -> float:
    return 0.5 * (1.0 + erf(z / sqrt(2.0)))

# Compute OR + 95% CI using log(OR) approx
# SE(log OR) ≈ sqrt(1/(U+pc) + 1/(V+pc)) :contentReference[oaicite:13]{index=13}
PC = 0.5  # small pseudocount for stability

rows = []
for ing in all_ings:
    u = int(U.get(ing, 0))
    v = int(V.get(ing, 0))

    or_hat = (u + PC) / (v + PC)
    log_or = float(np.log(or_hat))
    se = float(np.sqrt(1.0/(u + PC) + 1.0/(v + PC)))

    # 95% CI
    ci_lo = float(np.exp(log_or - 1.96*se))
    ci_hi = float(np.exp(log_or + 1.96*se))

    # Two-sided p-value (normal approx)
    z = 0.0 if se == 0 else log_or / se
    p = 2.0 * (1.0 - norm_cdf(abs(float(z))))

    rows.append({
        "ingredient": ing,
        "U_case_not_ctrl": u,
        "V_ctrl_not_case": v,
        "OR": or_hat,
        "log_OR": log_or,
        "SE_log_OR": se,
        "CI95_low": ci_lo,
        "CI95_high": ci_hi,
        "p_value": p,
        "discordant_total": u + v,
    })

ingredient_stats = pd.DataFrame(rows).sort_values(["p_value", "OR"], ascending=[True, False]).reset_index(drop=True)

print("Ingredient-level discordant-pairs results:")
display(ingredient_stats.head(25))

# -----------------------------
# BH-FDR correction (Benjamini–Hochberg)
# so we only show signals above random noise :contentReference[oaicite:14]{index=14}
# -----------------------------
def bh_fdr(pvals: np.ndarray) -> np.ndarray:
    pvals = np.asarray(pvals, dtype=float)
    n = len(pvals)
    order = np.argsort(pvals)
    ranked = pvals[order]
    q = np.empty(n, dtype=float)

    prev = 1.0
    for i in range(n-1, -1, -1):
        rank = i + 1
        val = ranked[i] * n / rank
        prev = min(prev, val)
        q[i] = prev

    qvals = np.empty(n, dtype=float)
    qvals[order] = q
    return qvals

ingredient_stats["q_value_bh"] = bh_fdr(ingredient_stats["p_value"].values)

# Optional filter: minimum discordant evidence (keeps UI clean for MVP)
MIN_DISCORDANT = 3
filtered_ingredient_stats = ingredient_stats[ingredient_stats["discordant_total"] >= MIN_DISCORDANT].copy()

print(f"Filtered (discordant_total >= {MIN_DISCORDANT}):", len(filtered_ingredient_stats))
display(filtered_ingredient_stats.head(25))

# -----------------------------
# Map to family and aggregate at family level
# Aggregate U and V across member ingredients -> family OR
# -----------------------------
if "ingredient_to_group" not in globals():
    raise ValueError("Expected `ingredient_to_group` mapping. Run your ingredient_groups.csv loading cell first.")

ingredient_stats["family"] = ingredient_stats["ingredient"].map(lambda x: ingredient_to_group.get(x, x))

family_uv = (
    ingredient_stats
      .groupby("family", as_index=False)
      .agg(
          U=("U_case_not_ctrl", "sum"),
          V=("V_ctrl_not_case", "sum"),
          n_members=("ingredient", "count"),
          discordant_total=("discordant_total", "sum"),
      )
)

# Compute family OR + CI + p in the same way
fam_rows = []
for _, r in family_uv.iterrows():
    u = int(r["U"]); v = int(r["V"])
    or_hat = (u + PC) / (v + PC)
    log_or = float(np.log(or_hat))
    se = float(np.sqrt(1.0/(u + PC) + 1.0/(v + PC)))
    ci_lo = float(np.exp(log_or - 1.96*se))
    ci_hi = float(np.exp(log_or + 1.96*se))
    z = 0.0 if se == 0 else log_or / se
    p = 2.0 * (1.0 - norm_cdf(abs(float(z))))

    fam_rows.append({
        "family": r["family"],
        "U_case_not_ctrl": u,
        "V_ctrl_not_case": v,
        "OR": or_hat,
        "CI95_low": ci_lo,
        "CI95_high": ci_hi,
        "p_value": p,
        "discordant_total": int(r["discordant_total"]),
        "n_members": int(r["n_members"]),
    })

family_stats = pd.DataFrame(fam_rows).sort_values(["p_value", "OR"], ascending=[True, False]).reset_index(drop=True)
family_stats["q_value_bh"] = bh_fdr(family_stats["p_value"].values)

# Clean MVP output: rank + trigger_strength from OR (monotonic mapping)
# We will use log(OR) clipped to positive part for "trigger strength"
family_stats["log_OR"] = np.log(family_stats["OR"])
family_stats["trigger_strength"] = family_stats["log_OR"].clip(lower=0)

# Rank by trigger_strength, then q_value
family_stats = family_stats.sort_values(["trigger_strength", "q_value_bh"], ascending=[False, True]).reset_index(drop=True)
family_stats["rank"] = family_stats["trigger_strength"].rank(ascending=False, method="dense").astype(int)

print("Family-level triggers (recommended primary output):")
display(family_stats.head(25))

# Optionally: show only BH-significant + evidence-backed families
Q_THRESH = 0.10
final_families = family_stats[
    (family_stats["q_value_bh"] <= Q_THRESH) &
    (family_stats["discordant_total"] >= MIN_DISCORDANT)
].copy()

print(f"Final families (q <= {Q_THRESH}, discordant_total >= {MIN_DISCORDANT}):", len(final_families))
display(final_families[[
    "rank","family","trigger_strength","OR","CI95_low","CI95_high",
    "U_case_not_ctrl","V_ctrl_not_case","discordant_total","q_value_bh","n_members"
]].head(50))


Strata built: 4
Controls per event (min/median/max): 3 3.5 4


,event_id,event_time,case_start,case_end,control_windows
0,0,2026-01-02 22:30:00,2026-01-01 22:30:00,2026-01-02 22:30:00,"[(2025-12-30 22:30:00, 2025-12-31 22:30:00), (..."
1,1,2026-01-06 22:40:00,2026-01-05 22:40:00,2026-01-06 22:40:00,"[(2026-01-03 22:40:00, 2026-01-04 22:40:00), (..."
2,2,2026-01-11 23:10:00,2026-01-10 23:10:00,2026-01-11 23:10:00,"[(2026-01-08 23:10:00, 2026-01-09 23:10:00), (..."
3,3,2026-01-16 22:55:00,2026-01-15 22:55:00,2026-01-16 22:55:00,"[(2026-01-13 22:55:00, 2026-01-14 22:55:00), (..."


,event_id,event_time,case_set,control_union_set
0,0,2026-01-02 22:30:00,"{milk, chicken, egg, beans, rice, carrot, pure...","{pepper, rice, lettuce, butter, apple, cucumbe..."
1,1,2026-01-06 22:40:00,"{pepper, potato, rice, egg, beans, lettuce, oa...","{rice, lettuce, butter, apple, cucumber, honey..."
2,2,2026-01-11 23:10:00,"{cream, tomato, chicken, lettuce, beans, apple...","{rice, lettuce, apple, cucumber, honey, potato..."
3,3,2026-01-16 22:55:00,"{milk, potato, lettuce, onion, cheese, carrot,...","{pepper, rice, lettuce, apple, cucumber, honey..."


Ingredient-level discordant-pairs results:


,ingredient,U_case_not_ctrl,V_ctrl_not_case,OR,log_OR,SE_log_OR,CI95_low,CI95_high,p_value,discordant_total
0,apple,0,3,0.142857,-1.945910,1.511858,0.007379,2.765804,0.198060,3
1,onion,0,3,0.142857,-1.945910,1.511858,0.007379,2.765804,0.198060,3
2,pasta,0,3,0.142857,-1.945910,1.511858,0.007379,2.765804,0.198060,3
3,tomato,0,3,0.142857,-1.945910,1.511858,0.007379,2.765804,0.198060,3
4,cheese,2,0,5.000000,1.609438,1.549193,0.240032,104.152571,0.298857,2
5,butter,0,2,0.200000,-1.609438,1.549193,0.009601,4.166103,0.298857,2
6,chicken,0,2,0.200000,-1.609438,1.549193,0.009601,4.166103,0.298857,2
7,cucumber,0,2,0.200000,-1.609438,1.549193,0.009601,4.166103,0.298857,2
8,oats,0,2,0.200000,-1.609438,1.549193,0.009601,4.166103,0.298857,2
9,rice,0,2,0.200000,-1.609438,1.549193,0.009601,4.166103,0.298857,2


Filtered (discordant_total >= 3): 8


,ingredient,U_case_not_ctrl,V_ctrl_not_case,OR,log_OR,SE_log_OR,CI95_low,CI95_high,p_value,discordant_total,q_value_bh
0,apple,0,3,0.142857,-1.945910,1.511858,0.007379,2.765804,0.198060,3,0.619006
1,onion,0,3,0.142857,-1.945910,1.511858,0.007379,2.765804,0.198060,3,0.619006
2,pasta,0,3,0.142857,-1.945910,1.511858,0.007379,2.765804,0.198060,3,0.619006
3,tomato,0,3,0.142857,-1.945910,1.511858,0.007379,2.765804,0.198060,3,0.619006
17,milk,2,1,1.666667,0.510826,1.032796,0.220148,12.617755,0.620879,3,0.620879
18,pepper,1,2,0.600000,-0.510826,1.032796,0.079253,4.542392,0.620879,3,0.620879
19,potato,1,2,0.600000,-0.510826,1.032796,0.079253,4.542392,0.620879,3,0.620879
20,pure olive oil,1,2,0.600000,-0.510826,1.032796,0.079253,4.542392,0.620879,3,0.620879


Family-level triggers (recommended primary output):


,family,U_case_not_ctrl,V_ctrl_not_case,OR,CI95_low,CI95_high,p_value,discordant_total,n_members,q_value_bh,log_OR,trigger_strength,rank
0,cheese,2,0,5.000000,0.240032,104.152571,0.298857,2,1,0.619006,1.609438,1.609438,1
1,cream,1,0,3.000000,0.122205,73.646667,0.501100,1,1,0.619006,1.098612,1.098612,2
2,milk,2,1,1.666667,0.220148,12.617755,0.620879,3,1,0.620879,0.510826,0.510826,3
3,apple,0,3,0.142857,0.007379,2.765804,0.198060,3,1,0.619006,-1.945910,0.000000,4
4,onion,0,3,0.142857,0.007379,2.765804,0.198060,3,1,0.619006,-1.945910,0.000000,4
5,pasta,0,3,0.142857,0.007379,2.765804,0.198060,3,1,0.619006,-1.945910,0.000000,4
6,tomato,0,3,0.142857,0.007379,2.765804,0.198060,3,1,0.619006,-1.945910,0.000000,4
7,butter,0,2,0.200000,0.009601,4.166103,0.298857,2,1,0.619006,-1.609438,0.000000,4
8,chicken,0,2,0.200000,0.009601,4.166103,0.298857,2,1,0.619006,-1.609438,0.000000,4
9,cucumber,0,2,0.200000,0.009601,4.166103,0.298857,2,1,0.619006,-1.609438,0.000000,4


Final families (q <= 0.1, discordant_total >= 3): 0


,rank,family,trigger_strength,OR,CI95_low,CI95_high,U_case_not_ctrl,V_ctrl_not_case,discordant_total,q_value_bh,n_members


In [311]:
# ------------------------------------------------------------
# Export: recommended (discordant-pairs + BH-FDR) trigger tables
# ------------------------------------------------------------

from pathlib import Path
import pandas as pd

OUT_DIR = Path("../data/processed") / "trigger_detection"
OUT_DIR.mkdir(parents=True, exist_ok=True)

run_meta = {
    "method": "case_crossover_discordant_pairs_bh_fdr",
    "case_hours": CASE_HOURS,
    "control_day_lags": "|".join(map(str, CONTROL_DAY_LAGS)),
    "event_severity_threshold": EVENT_SEVERITY_THRESHOLD,
    "episode_gap_hours": EPISODE_GAP_HOURS,
    "pseudocount": PC,
    "min_discordant": MIN_DISCORDANT,
    "q_thresh": Q_THRESH,
}

# Ingredient-level table (all ingredients with U/V, OR, CI, p, q)
ingredient_out = ingredient_stats.copy()
for k, v in run_meta.items():
    ingredient_out[k] = v

# Family-level full table (ranked, with OR/CI/q + evidence)
family_out = family_stats.copy()
for k, v in run_meta.items():
    family_out[k] = v

# Family-level "final" (filtered by q and evidence) — what you'd show to users by default
final_out = final_families.copy()
for k, v in run_meta.items():
    final_out[k] = v

ingredient_path = OUT_DIR / "recommended_ingredient_stats.csv"
family_path     = OUT_DIR / "recommended_family_stats.csv"
final_path      = OUT_DIR / "recommended_final_families.csv"

ingredient_out.to_csv(ingredient_path, index=False)
family_out.to_csv(family_path, index=False)
final_out.to_csv(final_path, index=False)

display(pd.DataFrame([
    {"saved": "ingredient-level (recommended)", "rows": len(ingredient_out), "path": str(ingredient_path)},
    {"saved": "family-level (recommended)",     "rows": len(family_out),     "path": str(family_path)},
    {"saved": "final families (recommended)",   "rows": len(final_out),      "path": str(final_path)},
]))


,saved,rows,path
0,ingredient-level (recommended),21,../data/processed/trigger_detection/recommende...
1,family-level (recommended),21,../data/processed/trigger_detection/recommende...
2,final families (recommended),0,../data/processed/trigger_detection/recommende...
